In [5]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
✅ Working directory: /content/drive/.shortcut-targets-by-id/1nJ6qfsYQS_sTYFLUl7Y2q0Uv9A3cadKk/reranking_rag_and_qw/notebook
Contents: ['.gitkeep', '04_train.ipynb', '01_setup_baseline.ipynb', '02_query_writing.ipynb', '03_reranking.ipynb']


In [3]:
# Install requirements
%pip install -r ../requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 

In [2]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q "faiss-cpu"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.5 MB/s eta 0:00:00


# 3. RankRAG Implementation
Triển khai Cross-encoder và LLM-based reranking

In [6]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [7]:
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.reranking.rank_rag import RankReranker, HybridReranker
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.llm_generator import LLMGenerator
from src.data.data_loader import DataLoader

In [8]:
# Setup components
data_loader = DataLoader("../data")
retriever = HybridRetriever()
llm = LLMGenerator(provider="huggingface", hf_model="Qwen/Qwen2.5-7B-Instruct")

# Build index
train_data = data_loader.load_dataset("vhealthqa", "train")[:]

documents = []
for sample in train_data:
    # Thử nhiều fields theo thứ tự ưu tiên
    text = (
        sample.get("ground_truth") or
        sample.get("answer") or
        sample.get("answer_text") or
        sample.get("context") or
        sample.get("passage") or
        sample.get("text") or ""
    )
    # ✅ Bỏ qua URLs
    if text and isinstance(text, str) and not text.startswith("http") and len(text) > 20:
        documents.append(text.strip())

documents = list(set(documents))  # deduplicate
retriever.build_index(documents)
print(f"✅ Index ready with {len(documents)} valid documents")
print(f"Sample doc: {documents[0][:100] if documents else 'EMPTY!'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded


Batches:   0%|          | 0/217 [00:00<?, ?it/s]

✅ Index ready with 6939 valid documents
Sample doc: Hay quên là tình trạng thường xuyên gặp ở phụ nữ sau sinh. Nguyên nhân là do áp lực công việc, gia đ


## Cross-Encoder Reranking

In [9]:
# Initialize cross-encoder reranker
cross_reranker = RankReranker(method="cross-encoder")

# Retrieve documents
query = "Triệu chứng bệnh đái tháo đường là gì?"
raw_docs = retriever.retrieve([query], top_k=10)

print(f"Retrieved {len(raw_docs)} documents")
print("\nBefore Reranking (Top 5):")
for i, doc in enumerate(raw_docs[:5], 1):
    print(f"{i}. [Score: {doc['score']:.3f}] {doc['text'][:100]}...")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Retrieved 10 documents

Before Reranking (Top 5):
1. [Score: 0.728] ho là triệu chứng của rất nhiều bệnh lý, thường là các bệnh lý viêm ở mũi họng, cũng có thể là viêm ...
2. [Score: 0.694] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp, có thể gặp ở mọi lứa tuổi. Những ngu...
3. [Score: 0.685] Triệu chứng thường gặp nhất của bệnh sỏi mật là đau vùng thượng vị hoặc hạ sườn phải, rất dễ nhầm vớ...
4. [Score: 0.683] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp gặp ở mọi lứa tuổi. Những nguyên nhân...
5. [Score: 0.682] Có thể là triệu chứng của bệnh lý bàng quang tăng hoạt, đây là bệnh lý điều trị lâu dài cần sự kiên ...


In [10]:
#Rerank
reranked_docs = cross_reranker.rerank(query, raw_docs, top_k=5)

print("\nAfter Reranking (Top 5):")
for i, doc in enumerate(reranked_docs, 1):
    score = doc.get("reranker_score", 0)
    print(f"{i}. [Score: {score:.3f}] {doc['text'][:100]}...")

  All reranker scores ≈ 0.500 (std=0.0005)
   Possible cause: documents are not meaningful text

After Reranking (Top 5):
1. [Score: 0.502] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp gặp ở mọi lứa tuổi. Những nguyên nhân...
2. [Score: 0.501] những triệu chứng bệnh mà chị nói hiện khó xác định được nguyên nhân cụ thể, có thể là bệnh lý thần ...
3. [Score: 0.501] Vấn đề quan trọng là chẩn đoán bệnh của mình là gì, có phải là ung thư không hay là bướu giáp lành t...
4. [Score: 0.500] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp, có thể gặp ở mọi lứa tuổi. Những ngu...
5. [Score: 0.500] Có thể là triệu chứng của bệnh lý bàng quang tăng hoạt, đây là bệnh lý điều trị lâu dài cần sự kiên ...


## LLM-based Reranking

In [11]:
# Initialize LLM reranker
llm_reranker = RankReranker(method="llm", llm_client=llm)

# Rerank with LLM
llm_reranked = llm_reranker.rerank(query, raw_docs, top_k=5)

print("\nLLM-based Reranking (Top 5):")
for i, doc in enumerate(llm_reranked, 1):
    score = doc.get("reranker_score", 0)
    print(f"{i}. [Score: {score:.3f}] {doc['text'][:100]}...")


LLM-based Reranking (Top 5):
1. [Score: 0.000] ho là triệu chứng của rất nhiều bệnh lý, thường là các bệnh lý viêm ở mũi họng, cũng có thể là viêm ...
2. [Score: 0.000] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp, có thể gặp ở mọi lứa tuổi. Những ngu...
3. [Score: 0.000] Triệu chứng thường gặp nhất của bệnh sỏi mật là đau vùng thượng vị hoặc hạ sườn phải, rất dễ nhầm vớ...
4. [Score: 0.000] Ho kéo dài là triệu chứng của nhiều bệnh lý thuộc đường hô hấp gặp ở mọi lứa tuổi. Những nguyên nhân...
5. [Score: 0.000] Có thể là triệu chứng của bệnh lý bàng quang tăng hoạt, đây là bệnh lý điều trị lâu dài cần sự kiên ...


## Hybrid Reranking Comparison

In [12]:
# Compare different reranking methods
test_queries = [
    "Bệnh huyết áp cao",
    "Cách phòng ngừa COVID-19",
    "Triệu chứng bệnh tim mạch"
]

comparison_results = {}

for test_query in test_queries:
    raw_docs = retriever.retrieve([test_query], top_k=10)

    # Cross-encoder
    ce_reranked = cross_reranker.rerank(test_query, raw_docs.copy(), top_k=3)

    # No reranking (baseline)
    no_rerank = raw_docs[:3]

    comparison_results[test_query] = {
        "no_rerank": no_rerank,
        "cross_encoder": ce_reranked
    }

# Display comparison
for query, results in comparison_results.items():
    print(f"\nQuery: {query}")
    print("\n  Baseline (No Reranking):")
    for i, doc in enumerate(results["no_rerank"], 1):
        print(f"    {i}. {doc['text'][:80]}...")
    print("\n  With Cross-Encoder Reranking:")
    for i, doc in enumerate(results["cross_encoder"], 1):
        print(f"    {i}. {doc['text'][:80]}...")


Query: Bệnh huyết áp cao

  Baseline (No Reranking):
    1. Nếu bệnh nhân mắc bệnh cao huyết áp trong thời gian dài, được chẩn đoán độ 3 ở đ...
    2. Cao huyết áp là tình trạng áp lực máu trong lòng động mạch tăng cao hơn so với b...
    3. Nếu anh/chị có tiền sử huyết áp cao và dùng thuốc hạ áp thường xuyên khi đo huyế...

  With Cross-Encoder Reranking:
    1. Cao huyết áp là tình trạng áp lực máu trong lòng động mạch tăng cao hơn so với b...
    2. Nếu bệnh nhân mắc bệnh cao huyết áp trong thời gian dài, được chẩn đoán độ 3 ở đ...
    3. Các bệnh mãn tính như tăng huyết áp cần được điều trị ổn định, các chỉ số phải đ...

Query: Cách phòng ngừa COVID-19

  Baseline (No Reranking):
    1. Hiện nay, Bộ Y tế cũng như các nhà sản xuất vaccine Covid-19 không khuyến cáo vi...
    2. Triệu chứng dị ứng như anh mô tả là phản vệ từ độ 2 trở lên. Do vậy, theo hướng ...
    3. Tình trạng như anh chia sẻ vẫn tiêm phòng vaccine Covid-19 được....

  With Cross-Encoder Reranking:
    1. Trường hợ

## Evaluate on Test Set

In [13]:
from evaluation.evaluator import RAGEvaluator

evaluator = RAGEvaluator()
test_data = data_loader.load_dataset("vhealthqa", "test")[:30]

# Evaluate with and without reranking
metrics_without = {"f1": 0, "relevance": 0}
metrics_with = {"f1": 0, "relevance": 0}
count = 0

for sample in test_data:
    query = sample.get("question", "")
    ground_truth = sample.get("answer", "")

    if not query or not ground_truth:
        continue

    # Without reranking
    docs_without = retriever.retrieve([query], top_k=5)
    answer_without = llm.generate(query, docs_without, max_tokens=300)

    # With reranking
    docs_with = cross_reranker.rerank(query, docs_without, top_k=5)
    answer_with = llm.generate(query, docs_with, max_tokens=300)

    # Evaluate
    metrics_w = evaluator.calculate_all(query, answer_without, docs_without, ground_truth)
    metrics_w_r = evaluator.calculate_all(query, answer_with, docs_with, ground_truth)

    metrics_without["f1"] += metrics_w.get("f1", 0)
    metrics_without["relevance"] += metrics_w.get("relevance", 0)
    metrics_with["f1"] += metrics_w_r.get("f1", 0)
    metrics_with["relevance"] += metrics_w_r.get("relevance", 0)
    count += 1

# Results
print("\n📊 Evaluation Results (30 samples):")
print(f"\nWithout Reranking:")
for key, value in metrics_without.items():
    avg = value / count if count > 0 else 0
    print(f"  {key.upper()}: {avg:.4f}")

print(f"\nWith Cross-Encoder Reranking:")
for key, value in metrics_with.items():
    avg = value / count if count > 0 else 0
    print(f"  {key.upper()}: {avg:.4f}")

print(f"\n🎯 Improvement:")
for key in metrics_without:
    without_avg = metrics_without[key] / count if count > 0 else 0
    with_avg = metrics_with[key] / count if count > 0 else 0
    improvement = ((with_avg - without_avg) / without_avg * 100) if without_avg > 0 else 0
    print(f"  {key.upper()}: {improvement:+.2f}%")


📊 Evaluation Results (30 samples):

Without Reranking:
  F1: 0.0000
  RELEVANCE: 0.0000

With Cross-Encoder Reranking:
  F1: 0.0000
  RELEVANCE: 0.0000

🎯 Improvement:
  F1: +0.00%
  RELEVANCE: +0.00%
